# Stage 10 — Governance Adaptation

This notebook implements the post-hoc governance adjustment to address the senior investigator's feedback. It discounts priority scores for cases whose anomaly signals are purely administrative, while preserving rankings for cases with genuine financial leakages.

In [1]:
import pandas as pd
import numpy as np
import os
import sys

# Ensure we can import from src/
sys.path.append(os.path.abspath('../'))
from src.governance import apply_governance_adjustment

print("Packages imported successfully.")

Packages imported successfully.


In [2]:
# 1. Load the scored cases
df_scored = pd.read_csv('../data/processed/scored_cases.csv')
print(f"Loaded {len(df_scored)} cases.")

Loaded 4200 cases.


In [3]:
# 2. Apply governance adjustment
df_gov = apply_governance_adjustment(df_scored)
print("Governance adjustment applied.")

Governance adjustment applied.


In [4]:
# 3. Establish deterministic new ranking
# Sort by adjusted_priority_score descending, and case_id ascending as a tie-breaker
df_gov = df_gov.sort_values(by=['adjusted_priority_score', 'case_id'], ascending=[False, True]).reset_index(drop=True)
df_gov['adjusted_rank'] = df_gov.index + 1
print("New ranking established.")

New ranking established.


In [5]:
# 4. Specifically report C-33248 metrics
df_scored_sorted = df_scored.sort_values(by=['priority_score', 'case_id'], ascending=[False, True]).reset_index(drop=True)
df_scored_sorted['original_rank'] = df_scored_sorted.index + 1

original_c = df_scored_sorted[df_scored_sorted['case_id'] == 'C-33248']
adjusted_c = df_gov[df_gov['case_id'] == 'C-33248']

orig_rank = original_c['original_rank'].values[0]
orig_score = original_c['priority_score'].values[0]
adj_rank = adjusted_c['adjusted_rank'].values[0]
adj_score = adjusted_c['adjusted_priority_score'].values[0]

print("=== Case C-33248 Verification ===")
print(f"Original Rank: {orig_rank}")
print(f"Adjusted Rank: {adj_rank}")
print(f"Original Priority Score: {orig_score:.6f}")
print(f"Adjusted Priority Score: {adj_score:.6f}")

=== Case C-33248 Verification ===
Original Rank: 485
Adjusted Rank: 1097
Original Priority Score: 0.388189
Adjusted Priority Score: 0.200565


In [6]:
# 5. Compare Top cuts before and after
orig_t20 = set(df_scored_sorted.head(20)['case_id'])
new_t20 = set(df_gov.head(20)['case_id'])

orig_t50 = set(df_scored_sorted.head(50)['case_id'])
new_t50 = set(df_gov.head(50)['case_id'])

orig_t100 = set(df_scored_sorted.head(100)['case_id'])
new_t100 = set(df_gov.head(100)['case_id'])

print(f"Top 20 cases affected: {len(orig_t20 - new_t20)}")
print(f"Top 50 cases affected: {len(orig_t50 - new_t50)}")
print(f"Top 100 cases affected: {len(orig_t100 - new_t100)}")

print("\nFinal Top 20 Cases:")
print(df_gov[['adjusted_rank', 'case_id', 'adjusted_priority_score', 'priority_score', 'contact_attempts', 'payment_adjustments', 'total_excess_amount']].head(20))

Top 20 cases affected: 3
Top 50 cases affected: 8
Top 100 cases affected: 19

Final Top 20 Cases:
    adjusted_rank  case_id  adjusted_priority_score  priority_score  \
0               1  C-33263                 1.000000        1.000000   
1               2  C-30644                 0.957167        0.957167   
2               3  C-33980                 0.917495        0.917495   
3               4  C-31298                 0.882768        0.882768   
4               5  C-30945                 0.881573        0.881573   
5               6  C-30824                 0.871548        0.871548   
6               7  C-34118                 0.863800        0.863800   
7               8  C-32962                 0.841570        0.841570   
8               9  C-32035                 0.837788        0.837788   
9              10  C-33702                 0.834542        0.834542   
10             11  C-30387                 0.824103        0.824103   
11             12  C-30121                 0.81265

In [7]:
# 6. Verify that genuine financial anomaly cases are preserved
new_t100_df = df_gov.head(100)
financial_anomalies_in_t100 = new_t100_df[
    (new_t100_df['post_closure_payment_count'] > 0) |
    (new_t100_df['has_same_month_multi_payments'] == 1) |
    (new_t100_df['total_excess_amount'] > 200.0)
]
print(f"Number of cases in new Top 100 with active financial anomalies: {len(financial_anomalies_in_t100)} / 100")

Number of cases in new Top 100 with active financial anomalies: 82 / 100


In [8]:
# 7. Verify that demographic fields are not used in governance scoring
demographics = ['district', 'age_band', 'language_preference', 'tenure']
for d in demographics:
    grouped = df_gov.groupby(d)['adjusted_priority_score'].mean()
    print(f"\nAverage adjusted priority score by {d}:")
    print(grouped)


Average adjusted priority score by district:
district
Ash Hill          0.186820
Calder Central    0.175169
Northgate         0.179423
Weybridge         0.165651
Name: adjusted_priority_score, dtype: float64

Average adjusted priority score by age_band:
age_band
18-29    0.178506
30-44    0.176159
45-59    0.166011
60-74    0.182595
75+      0.179567
Name: adjusted_priority_score, dtype: float64

Average adjusted priority score by language_preference:
language_preference
English    0.175447
Other      0.179023
Spanish    0.176414
Name: adjusted_priority_score, dtype: float64

Average adjusted priority score by tenure:
tenure
No fixed abode          0.180691
Owner-occupier          0.181191
Private tenancy         0.170226
Resident with family    0.177059
Social tenancy          0.170664
Name: adjusted_priority_score, dtype: float64


In [9]:
# 8. Save governed and final Top 20 cases
os.makedirs('../data/processed', exist_ok=True)
df_gov.to_csv('../data/processed/governed_cases.csv', index=False)
df_gov.head(20).to_csv('../data/processed/final_top20_cases.csv', index=False)
print("Saved governed_cases.csv and final_top20_cases.csv successfully.")

Saved governed_cases.csv and final_top20_cases.csv successfully.


In [10]:
# 9. Generate and save final Top 20 explanations
from src.explainability import generate_governed_case_explanations
df_explanations = generate_governed_case_explanations(df_gov)
df_explanations.head(20).to_csv('../data/processed/final_top20_explanations.csv', index=False)
print("Saved final_top20_explanations.csv successfully.")
print(df_explanations[['rank', 'case_id', 'top_contributing_signals', 'plain_language_explanation']].head(20))

Saved final_top20_explanations.csv successfully.
    rank  case_id                           top_contributing_signals  \
0      1  C-33263          high_award_excess | high_contact_attempts   
1      2  C-30644          high_award_excess | high_contact_attempts   
2      3  C-33980        multi_payment_month | high_contact_attempts   
3      4  C-31298          high_award_excess | high_contact_attempts   
4      5  C-30945       post_closure_payment | high_contact_attempts   
5      6  C-30824                               post_closure_payment   
6      7  C-34118          high_award_excess | high_contact_attempts   
7      8  C-32962                                  high_award_excess   
8      9  C-32035  multi_payment_month | high_award_excess | high...   
9     10  C-33702  post_closure_payment | high_award_excess | hig...   
10    11  C-30387       post_closure_payment | unreviewed_review_gap   
11    12  C-30121  post_closure_payment | high_contact_attempts |...   
12    13  C-311

In [11]:
# 10. Run governed fairness analysis and save report
from src.fairness import analyze_fairness
df_fairness = analyze_fairness(
    scored_path='../data/processed/governed_cases.csv',
    top20_path='../data/processed/final_top20_cases.csv',
    score_col='adjusted_priority_score'
)
df_fairness.to_csv('../data/processed/final_fairness_report.csv', index=False)
print("Saved final_fairness_report.csv successfully.")
print(df_fairness.head(10))

Saved final_fairness_report.csv successfully.
     demographic_field           group  population_count  \
0             district        Ash Hill               663   
1             district  Calder Central              1433   
2             district       Northgate              1121   
3             district       Weybridge               983   
4             age_band           18-29               701   
5             age_band           30-44              1112   
6             age_band           45-59               992   
7             age_band           60-74               874   
8             age_band             75+               521   
9  language_preference         English              3277   

   population_percentage  top20_count  top20_percentage  representation_ratio  \
0              15.785714            7              35.0              2.217195   
1              34.119048            6              30.0              0.879274   
2              26.690476            5             

## Surprise Challenge / Governance Adaptation Summary

* **Cap Selection**: A maximum administrative dampening cap of **60%** was selected based on sensitivity analysis as the smallest tested cap that:
  1. Reduced the number of administrative-only cases in the Top 20 to **zero**.
  2. Preserved the evaluated genuine financial anomaly cases (`C-34196`, `C-33728`, and `C-30954` in the Top 20).
  3. Retained at least **40%** of the original Isolation Forest score to preserve statistical auditability.
  4. Demoted target case **C-33248** appropriately to rank **1,097**.
* **Empirical Threshold**: Defined **$200** as an empirical threshold in this governance analysis to protect cases with minor excess payments.
* **Governance Rule**: For administrative-only cases (no post-closure payments, same-month duplicates, and excess payments <= $200), scores are scaled by `1 - 0.60 * admin_intensity`. Cases with active financial anomaly signals keep their original priority score.
* **Demographic Exclusion**: Demographic fields were completely excluded from the governance scoring adjustments to prevent direct bias.
